# 📦 Supply Chain Demand Forecasting
## Using Temporal Fusion Transformer (TFT)
TFT is a state-of-the-art attention-based deep learning model for multi-horizon time series forecasting.
It handles:
- **Static covariates** (e.g., product category)
- **Known future inputs** (e.g., holidays, promotions)
- **Past observed inputs** (e.g., historical demand)
- **Interpretable attention weights**

In [ ]:
# ==========================
# STEP 1: INSTALL DEPENDENCIES
# ==========================
!pip install pytorch-forecasting pytorch-lightning lightning --quiet

In [ ]:
# ==========================
# STEP 2: IMPORTS
# ==========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import MAE, RMSE, QuantileLoss

print('✅ All libraries imported successfully!')

In [ ]:
# ==========================
# STEP 3: GENERATE / LOAD DATA
# ==========================
# Synthetic supply chain demand data
# Replace this block with your actual dataset

np.random.seed(42)
n_products = 3
n_timesteps = 200

data_list = []
for prod_id in range(n_products):
    time_idx = np.arange(n_timesteps)
    trend     = 100 + prod_id * 20 + 0.3 * time_idx
    seasonality = 30 * np.sin(2 * np.pi * time_idx / 12)
    noise     = np.random.normal(0, 10, n_timesteps)
    demand    = trend + seasonality + noise

    df_prod = pd.DataFrame({
        'time_idx':       time_idx,
        'product_id':     str(prod_id),
        'demand':         demand,
        # Known future covariates
        'month':          (time_idx % 12).astype(str),
        'is_promotion':   (time_idx % 10 == 0).astype(int).astype(str),
        # Static covariate
        'product_category': 'cat_' + str(prod_id % 2),
    })
    data_list.append(df_prod)

data = pd.concat(data_list, ignore_index=True)
print(f'✅ Dataset shape: {data.shape}')
data.head()

In [ ]:
# ==========================
# STEP 4: CONFIGURE TFT DATASET
# ==========================
max_prediction_length = 12   # Forecast 12 steps ahead
max_encoder_length    = 36   # Use past 36 steps as context

training_cutoff = data['time_idx'].max() - max_prediction_length

training = TimeSeriesDataSet(
    data[data['time_idx'] <= training_cutoff],
    time_idx                = 'time_idx',
    target                  = 'demand',
    group_ids               = ['product_id'],
    min_encoder_length      = max_encoder_length // 2,
    max_encoder_length      = max_encoder_length,
    min_prediction_length   = 1,
    max_prediction_length   = max_prediction_length,
    static_categoricals     = ['product_id', 'product_category'],
    time_varying_known_categoricals  = ['month', 'is_promotion'],
    time_varying_unknown_reals       = ['demand'],
    target_normalizer       = GroupNormalizer(groups=['product_id'], transformation='softplus'),
    add_relative_time_idx   = True,
    add_target_scales       = True,
    add_encoder_length      = True,
)

validation = TimeSeriesDataSet.from_dataset(
    training, data, predict=True, stop_randomization=True
)

train_dataloader = training.to_dataloader(train=True,  batch_size=64, num_workers=0)
val_dataloader   = validation.to_dataloader(train=False, batch_size=64, num_workers=0)

print('✅ Datasets & DataLoaders ready!')

In [ ]:
# ==========================
# STEP 5: BUILD TFT MODEL
# ==========================
pl.seed_everything(42)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate           = 0.03,
    hidden_size             = 64,        # TFT hidden layer size
    attention_head_size     = 4,         # Multi-head attention heads
    dropout                 = 0.1,
    hidden_continuous_size  = 16,
    loss                    = QuantileLoss(),   # Probabilistic forecasting
    log_interval            = 10,
    reduce_on_plateau_patience = 4,
)

print(f'✅ TFT Model built!')
print(f'   Number of parameters: {tft.size()/1e3:.1f}k')

In [ ]:
# ==========================
# STEP 6: TRAIN THE MODEL
# ==========================
early_stop  = EarlyStopping(monitor='val_loss', min_delta=1e-4, patience=10, mode='min')
lr_logger   = LearningRateMonitor()
logger      = TensorBoardLogger('lightning_logs')

trainer = pl.Trainer(
    max_epochs          = 50,
    accelerator         = 'auto',
    enable_model_summary = True,
    gradient_clip_val   = 0.1,
    callbacks           = [early_stop, lr_logger],
    logger              = logger,
    enable_progress_bar = True,
)

trainer.fit(
    tft,
    train_dataloaders = train_dataloader,
    val_dataloaders   = val_dataloader,
)

print('✅ Training complete!')

In [ ]:
# ==========================
# STEP 7: EVALUATE & PREDICT
# ==========================
best_model_path = trainer.checkpoint_callback.best_model_path
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path)

# Get predictions
predictions = best_tft.predict(val_dataloader, return_y=True, trainer_kwargs=dict(accelerator='auto'))

# Compute metrics
mae_val  = MAE()(predictions.output,  predictions.y)
rmse_val = RMSE()(predictions.output, predictions.y)

print(f'MAE  (TFT): {mae_val:.4f}')
print(f'RMSE (TFT): {rmse_val:.4f}')

In [ ]:
# ==========================
# STEP 8: PLOT RESULTS
# ==========================
raw_predictions = best_tft.predict(
    val_dataloader,
    mode='raw',
    return_x=True
)

# Plot each product
for idx in range(n_products):
    best_tft.plot_prediction(
        raw_predictions.x,
        raw_predictions.output,
        idx=idx,
        add_loss_to_title=True
    )
    plt.title(f'Supply Chain Demand Forecasting — Product {idx} (TFT)')
    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================
# STEP 9: INTERPRETABILITY
# ==========================
# TFT gives you feature importance for FREE!

interpretation = best_tft.interpret_output(
    raw_predictions.output, reduction='sum'
)

best_tft.plot_interpretation(interpretation)
plt.suptitle('TFT Feature Importance & Attention', fontsize=14)
plt.tight_layout()
plt.show()

print('\n✅ Done! TFT provides:')
print('   - Probabilistic forecasts (quantile ranges)')
print('   - Variable importance scores')
print('   - Attention weights over time')